# Cell Tower Map — US AT&T (Ford Vehicle SIM Data)

Visualises which AT&T cell towers Ford vehicles connect to, using:
- **Handover events** from `/home/jovyan/data/stage/handover_events/` (parquet)
- **OpenCelliD MCC=310 CSV** for real GPS coordinates per cell tower

Run cells top-to-bottom. The final cell writes `cell_tower_map.html`.

In [ ]:
import pandas as pd
import glob
import folium
from folium.plugins import HeatMap, MarkerCluster
from pathlib import Path

TOWERS_CSV   = Path("/home/jovyan/data/310.csv/310.csv")
HANDOVER_DIR = Path("/home/jovyan/data/stage/handover_events")
OUTPUT_HTML  = Path("/home/jovyan/telco-poc/notebooks/cell_tower_map.html")

assert TOWERS_CSV.exists(), f"File not found: {TOWERS_CSV}"
print("✓ OpenCelliD file found")

## Step 1 — Load OpenCelliD tower coordinates (AT&T LTE only)

In [ ]:
cols = ['radio','mcc','net','area','cell','unit','lon','lat','range',
        'samples','changeable','created','updated','averageSignal']

towers = pd.read_csv(TOWERS_CSV, header=None, names=cols)
att_lte = towers[(towers['net'] == 410) & (towers['radio'] == 'LTE')].copy()
att_lte = att_lte[['cell','lat','lon','area','samples']].rename(columns={'cell':'cell_id','area':'tac'})

print(f"AT&T LTE towers in OpenCelliD: {len(att_lte):,}")
att_lte.head(3)

## Step 2 — Count handover events per cell_id across all parquet data

In [ ]:
files = sorted(glob.glob(str(HANDOVER_DIR / '**/*.parquet'), recursive=True))
print(f"Parquet files: {len(files)}")

dfs = [pd.read_parquet(f, columns=['cell_id','vehicle_id']) for f in files]
events = pd.concat(dfs, ignore_index=True)

# Count events and unique vehicles per cell
cell_stats = events.groupby('cell_id').agg(
    event_count=('vehicle_id', 'count'),
    vehicle_count=('vehicle_id', 'nunique')
).reset_index()

print(f"Total events:      {len(events):,}")
print(f"Unique cell_ids:   {cell_stats['cell_id'].nunique():,}")
cell_stats.sort_values('event_count', ascending=False).head(5)

## Step 3 — Join event counts with tower coordinates

In [ ]:
mapped = cell_stats.merge(att_lte, on='cell_id', how='inner')
mapped = mapped.dropna(subset=['lat','lon'])

total_cells = cell_stats['cell_id'].nunique()
matched = len(mapped)
coverage_pct = 100 * mapped['event_count'].sum() / cell_stats['event_count'].sum()

print(f"Cell IDs matched to coordinates: {matched:,} / {total_cells:,} ({100*matched/total_cells:.0f}%)")
print(f"Event coverage:                  {coverage_pct:.1f}% of all handover events")
print(f"Geographic extent:")
print(f"  Lat: {mapped['lat'].min():.3f} → {mapped['lat'].max():.3f}")
print(f"  Lon: {mapped['lon'].min():.3f} → {mapped['lon'].max():.3f}")
mapped.sort_values('event_count', ascending=False).head(5)

## Step 4 — Build interactive map

In [ ]:
center_lat = mapped['lat'].median()
center_lon = mapped['lon'].median()

m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles='CartoDB positron')

# --- Layer 1: Heatmap weighted by event count ---
heat_data = [
    [row.lat, row.lon, row.event_count]
    for row in mapped.itertuples()
]
HeatMap(
    heat_data,
    name='Activity Heatmap',
    min_opacity=0.3,
    radius=18,
    blur=12,
    max_zoom=10
).add_to(m)

# --- Layer 2: Clustered markers with popup detail ---
cluster = MarkerCluster(name='Cell Towers').add_to(m)

for row in mapped.itertuples():
    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=max(4, min(18, row.event_count ** 0.4)),
        color='#e63946',
        fill=True,
        fill_color='#e63946',
        fill_opacity=0.6,
        popup=folium.Popup(
            f"<b>Cell ID:</b> {row.cell_id}<br>"
            f"<b>TAC:</b> {row.tac}<br>"
            f"<b>Events:</b> {row.event_count:,}<br>"
            f"<b>Vehicles:</b> {row.vehicle_count}<br>"
            f"<b>Lat/Lon:</b> {row.lat:.4f}, {row.lon:.4f}",
            max_width=220
        ),
        tooltip=f"Cell {row.cell_id} — {row.event_count:,} events"
    ).add_to(cluster)

folium.LayerControl().add_to(m)

m.save(OUTPUT_HTML)
print(f"✓ Map saved to {OUTPUT_HTML}")
print(f"  {matched} towers plotted across the US")
m

## Step 5 — Top 20 most active towers

In [ ]:
import matplotlib.pyplot as plt

top20 = mapped.sort_values('event_count', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(
    top20['cell_id'].astype(str),
    top20['event_count'],
    color='#e63946',
    alpha=0.8
)
ax.invert_yaxis()
ax.set_xlabel('Handover Events')
ax.set_title('Top 20 Most Active AT&T Cell Towers (Ford Vehicle Data)')
ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=8)
plt.tight_layout()
plt.show()